# Using `kugupu` to calculate molecular coupling networks

This notebook demonstrates how to calculate molecular coupling between fragments, inspect the results and save and load these results to file.  These results files will be the basis of all further analysis done using the `kugupu` package.

This will require version 0.20.0 of MDAnalysis, and kugupu to be installed.

In [13]:
import MDAnalysis as mda
import kugupu as kgp
import sys
import numpy as np
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "last_expr"
# np.set_printoptions(threshold=sys.maxsize)
np.set_printoptions(threshold=30)  # only print up to 100 elements


Firstly we create an `MDAnalysis.Universe` object from our simulation files:

In [14]:
u = mda.Universe('datafiles/C6.data', 'datafiles/C6.dcd')

This system has 46,500 atoms in 250 different fragments.

In [15]:
print(u.atoms.n_atoms, len(u.atoms.fragments))

46500 250


Our dynamics simulation has 5 frames of results.

In [16]:
print(u.trajectory.n_frames)

5


To perform the coupling calculations our `Universe` will require bond information (for determining fragments) and element information (for the tight binding calculations) stored inside the `.names` attribute.

Our Lammps Data file did not include element symbols, so we can add these to the Universe now...

In [17]:
def add_names(u):
    # Guesses atom names based upon masses
    def approx_equal(x, y):
        return abs(x - y) < 0.1
    
    # mapping of atom mass to element
    massdict = {}
    for m in set(u.atoms.masses):
        for elem, elem_mass in mda.guesser.tables.masses.items():
            if approx_equal(m, elem_mass):
                massdict[m] = elem
                break
        else:
            raise ValueError
            
    u.add_TopologyAttr('names')
    for m, e in massdict.items():
        u.atoms[u.atoms.masses == m].names = e

add_names(u)

## Running the coupling matrix calculation

The coupling matrix between fragments is calculated using the `kgp.coupling_matrix` function.

Here we are calculating the coupling matrix for fragments in the Universe `u` where
- coupling is calculated between fragments with a closest approach of less than 5.0 Angstrom (`nn_cutoff`)
- coupling is calculated between the LUMO upwards (`state='lumo'`)
- one state per fragment is considered (`degeneracy=1`)
- we will analyse up to frame 3 (`stop=3`)

This function will (for each frame)
- identify which fragments are close enough to possibly be electronically coupled
- run a tight binding calculation between all pairs identified
- calculate the molecular coupling based on this tight binding calculation

In [18]:
res = kgp.coupling_matrix(u, nn_cutoff=5.0, state='lumo', degeneracy=1, stop=1)

2025-06-17T16:31:34.887960+0100 INFO Processing 1 frames
2025-06-17T16:31:34.889427+0100 INFO Processing frame 1 of 1
2025-06-17T16:31:34.965546+0100 INFO Finding dimers within 5.0, passed 250 fragments
2025-06-17T16:31:35.323316+0100 INFO Found 3282 dimers
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_

no shift
Hii is [[-2.14000000e+01 -0.00000000e+00 -0.00000000e+00 ... -2.86431388e-06
  -1.30869185e-05 -1.01392083e-05]
 [-0.00000000e+00 -1.14000000e+01 -0.00000000e+00 ...  1.21741386e-06
   3.48512307e-06  5.00767256e-06]
 [-0.00000000e+00 -0.00000000e+00 -1.14000000e+01 ...  2.19435160e-06
   1.16780105e-05  8.56680811e-06]
 ...
 [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -1.36000000e+01
  -3.31241261e+00 -4.01394025e+00]
 [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -0.00000000e+00
  -1.36000000e+01 -3.13330324e+00]
 [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -0.00000000e+00
  -0.00000000e+00 -1.36000000e+01]]
Hij is [[-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 ...
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]]
v is [[-1.53085762e-04]
 [-2.23731259e-03]
 [-1.75857389e-03]
 ...
 [-4.53368846e-07]
 [ 1.99432609e-06]
 [-2.49290860e-07]]
e is [-10.27936597]
v is

ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 368 and 365 (0.991499 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 235 and 215 (0.997515 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/mini

v is [[-2.84078164e-04]
 [-1.16665832e-02]
 [-6.73680601e-03]
 ...
 [-6.17539510e-07]
 [-2.14442233e-07]
 [ 1.32826338e-06]]
e is [-10.42728372]
for j bit the H_frag insert is [-10.42728372]
no shift
Hii is [[-2.14000000e+01 -0.00000000e+00 -0.00000000e+00 ... -2.86431388e-06
  -1.30869185e-05 -1.01392083e-05]
 [-0.00000000e+00 -1.14000000e+01 -0.00000000e+00 ...  1.21741386e-06
   3.48512307e-06  5.00767256e-06]
 [-0.00000000e+00 -0.00000000e+00 -1.14000000e+01 ...  2.19435160e-06
   1.16780105e-05  8.56680811e-06]
 ...
 [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -1.36000000e+01
  -3.31241261e+00 -4.01394025e+00]
 [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -0.00000000e+00
  -1.36000000e+01 -3.13330324e+00]
 [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -0.00000000e+00
  -0.00000000e+00 -1.36000000e+01]]
Hij is [[-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 ...
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0.

ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 235 and 215 (0.985405 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using defau

...

ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 304 and 303 (0.999701 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using defau

Hii is [[-21.4         -0.          -0.         ...  -0.          -0.
   -0.        ]
 [ -0.         -11.4         -0.         ...  -0.          -0.
   -0.        ]
 [ -0.          -0.         -11.4        ...  -0.          -0.
   -0.        ]
 ...
 [ -0.          -0.          -0.         ... -13.6         -3.43022983
   -3.8656008 ]
 [ -0.          -0.          -0.         ...  -0.         -13.6
   -3.76769621]
 [ -0.          -0.          -0.         ...  -0.          -0.
  -13.6       ]]
Hij is [[-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 ...
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]]
no shift
Hii is [[-21.4         -0.          -0.         ...  -0.          -0.
   -0.        ]
 [ -0.         -11.4         -0.         ...  -0.          -0.
   -0.        ]
 [ -0.          -0.         -11.4        ...  -0.          -0.
   -0.        ]
 ...
 [ -0.          -0.          -0.         ... -1

ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/mini

Hii is [[-21.4         -0.          -0.         ...  -0.          -0.
   -0.        ]
 [ -0.         -11.4         -0.         ...  -0.          -0.
   -0.        ]
 [ -0.          -0.         -11.4        ...  -0.          -0.
   -0.        ]
 ...
 [ -0.          -0.          -0.         ... -13.6         -3.97398713
   -4.62580247]
 [ -0.          -0.          -0.         ...  -0.         -13.6
   -4.08481802]
 [ -0.          -0.          -0.         ...  -0.          -0.
  -13.6       ]]
Hij is [[-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 ...
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]
 [-0. -0. -0. ... -0. -0. -0.]]
no shift
Hii is [[-21.4         -0.          -0.         ...  -0.          -0.
   -0.        ]
 [ -0.         -11.4         -0.         ...  -0.          -0.
   -0.        ]
 [ -0.          -0.         -11.4        ...  -0.          -0.
   -0.        ]
 ...
 [ -0.          -0.          -0.         ... -1

ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/mini

The `res` object is a namedtuple which contains all the data necessary to perform further analysis.
This object has various attributes which will not be briefly explained.

The `.frames` attribute records which frames from the trajectory were analysed.
This is useful to later cross reference data with the original MD trajectory data.

In [19]:
print(res.frames)

[0]


The `.degeneracy` attribute stores how many degenerate states were considered for each fragment.
This value will not change over time, so this array has shape `nfragments`.

In this example only a single state per fragment was considered. 

In [20]:
print(res.degeneracy)

[1 1 1 ... 1 1 1]


The `.H_frag` attribute contains the molecular coupling values, stored inside a 3d numpy array.
The first dimension is along the number of frames (quasi time axis),
while the other two move along fragments in the system.

For example `res.H_frag[0, 1, 71]` gives the coupling (in eV) between the 2nd and 13th fragments in the first frame.

In [21]:
print(res.H_frag.shape)

print(res.H_frag[0, 1, 71])

(1, 250, 250)
0.03601076420417102


In [26]:
res.H_frag.shape

(1, 250, 250)

Producing these results is often a time consuming part of the analysis,
therefore it is wise to save them to a file so you can come back to them later!

This can be done using the `kugupu.save_results` function, which will save the results to a hdf5 (compressed) format.

In [ ]:
# kgp.save_results('myresults.hdf5', res)

FileExistsError: [Errno 17] Unable to synchronously create file (unable to open file: name = 'myresults.hdf5', errno = 17, error message = 'File exists', flags = 15, o_flags = a02)

These results can then be retrieved again using the `kugupu.load_results` function:

In [ ]:
kgp.load_results('./myresults.hdf5')